<a href="https://colab.research.google.com/github/josedanielisidororeyes/Automatizaci-n-Inteligence-con-Whatsapp-Cloud-API/blob/main/Proyecto_Integrador_Curso_META_Chatbot_IBERO_RAG%2BFine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalación de herramientas

In [ ]:
!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()


login(token  =  userdata.get("HugginFace"))
print("Sesión de Hugging Face iniciada correctamente.")

En primer lugar se inicia la sesión con Hugging Face

# Carga de Modelo Base Ligero

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

Posteriormente, se carga  el modelo base TinyLlama-1.1B-Chat-v1.0 con su respectivo tokenizador.

# Base de Conocimiento para RAG

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Objeto y principios rectores: La política CO259 regula la asignación de apoyos educativos y becas de la IBERO Puebla bajo principios de equidad, talento, mérito, rendimiento académico y vulnerabilidad socioeconómica.",
    "Apoyo IBERO Puebla: Es un apoyo educativo otorgado considerando la situación socioeconómica del núcleo familiar del solicitante y principios de excelencia académica, autorizado por el Comité del Programa.",
    "Becas IBERO (Ley General de Educación): Consisten en la exención total o parcial del pago de cuotas de inscripción o colegiaturas. Su asignación o renovación no puede condicionarse a créditos, gravámenes ni servicios adicionales.",
    "Beneficio Familia IBERO: Es un apoyo educativo del 20% al 30% dirigido a aspirantes y estudiantes que tengan hermanos, hermanas, padres, madres o tutores legales que sean estudiantes inscritos o egresados de la IBERO Puebla.",
    "Beneficio por Egresada(o) IBERO/SUJ: Es un apoyo educativo del 10% al 20% dirigido a personas egresadas o a sus hijas e hijos, provenientes de la IBERO Puebla o de cualquier plantel del Sistema Universitario Jesuita (SUJ).",
    "Conserva tu apoyo IBERO Puebla: Apoyo educativo dirigido a personas egresadas de algún programa superior de la institución que ya contaban con apoyo al graduarse, aplicable para continuar estudios sin exceder el 50% del costo total.",
    "Certificados de Estudio (Pago Anticipado): Cubren la totalidad de colegiaturas e inscripciones en preparatoria o créditos en licenciatura al momento de hacerse efectivos, conforme a las condiciones fijadas al momento de su adquisición.",
    "Presupuesto y sostenibilidad: Para salvaguardar la estabilidad financiera de la universidad, el Comité del Programa Impulso y Becas determina la asignación de apoyos conforme al presupuesto fijado para cada periodo académico."
]
embeddings_documentos = modelo_embeddings.encode(documentos, normalize_embeddings= True)
print("Embeddings generados:", embeddings_documentos.shape)

A continuación, se genera una base de conocimientos con base en los lineamientos de otorgamiento de becas de IBERO Puebla para evitar alucinaciones y brindar respuestas veridicas para futuros prospectos.

In [ ]:
# Definición de función de recuperación

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta], normalize_embeddings=True)
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

buscar_fragmento("¿Tengo un hermano estudiando y quiero una beca?")

A continuación, se generan embeddings con la base de conocimiento del punto anterior y se crea una función que busque los fragmentos más similares a las preguntas del usuario con la finalidad de que el LLM pueda utilizarlos en sus respuestas.

# Fine-Tuning del Modelo con LoRA

In [ ]:
def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=60):
    mensajes = [{"role": "user", "content": pregunta}]
    prompt_formateado = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt_formateado, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

prompt_prueba  = "¿Cómo obtener una beca para licenciatura si tengo hermanos estudiando en IBERO Puebla?"

Se genera una función que procesa la pregunta del usuario y la adapta al formato de conversación esperado por el modelo.

In [ ]:
# Definición del conjunto de datos para Fine-Tuning
from datasets import Dataset

def formatear_ejemplo(pregunta, respuesta):
    mensajes = [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize =  False)

pares = [
    (
        "¿Tengo un hermano estudiando en IBERO Puebla y quiero una beca?",
        "Puedes solicitar el Beneficio Familia IBERO, que otorga entre un 20% y 30% de apoyo si tienes familiares directos inscritos o egresados."
    ),
    (
        "¿Qué pasa con mis estudios si fallece mi tutor financiero?",
        "El Plan de Protección por Defunción cubre el 100% de colegiaturas, créditos e inscripciones pendientes hasta concluir la preparatoria o licenciatura."
    ),
    (
        "¿Cuál es el porcentaje máximo de apoyo si egresé de licenciatura y quiero estudiar posgrado?",
        "Con el apoyo 'Conserva tu apoyo IBERO Puebla', puedes recibir hasta un máximo del 50% del costo total del posgrado."
    ),
    (
        "¿La Beca IBERO por Ley de Educación me obliga a hacer actividades extra?",
        "No, por normativa la asignación o renovación de la Beca IBERO no se condiciona a créditos, pagos adicionales ni labores obligatorias."
    ),
    (
        "¿Qué apoyo existe para estudiar la Maestría en Investigación Educativa?",
        "Existe la Beca MAGIS, cuyo porcentaje se determina según la convocatoria y el resultado de la entrevista panel de admisión."
    ),
    (
        "¿Quién autoriza ampliar el presupuesto de becas si ya no alcanza?",
        "Lo analiza la Dirección de Contraloría a petición del Comité y lo aprueba formalmente el Rector."
    ),
    (
        "¿Los estudiantes con apoyo en licenciatura tienen actividades obligatorias?",
        "Sí, deben participar en el Plan de Experiencia Formativa y en campañas de procuración de fondos en proporción a su porcentaje de apoyo."
    ),
    (
        "¿Hay descuento para egresados de otras universidades jesuitas en posgrado?",
        "Sí, la Beca Ignacio Ellacuría S.J. ofrece apoyo de hasta el 100% para docentes y egresados de la red AUSJAL, además del beneficio general por egresado SUJ."
    ),
]

ejemplos = [{"texto": formatear_ejemplo(p, r)} for p, r in pares]
dataset = Dataset.from_list(ejemplos)
dataset


Se genera conjuntos  pares de ejemplo los cuales serán usados para realizar el ajuste-fino del modelo. De esta manera, se controla el tono de acuerdo a los requerimientos de IBERO Puebla.

In [ ]:
# Probar el modelo base con el prompt de prueba antes de hacer fine tuning
respuesta_base  =  generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

Se genera ua prueba antes de realizar ajuste fino. Como se puede observar la respuesta es pobre (inventada).

In [ ]:
# Configuraión de LoRA
!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

Despues se definen los parametros para la realización de Fine-Tuning a través de Low Ranking Adaptation  LoRA. Al realizar adaptaciones en los modulos de consulta y valor dentro de los mecanismos de atención de los transformers se reduce el número de parametros a entrenar.

In [ ]:
# Entranamiento con LoRA

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_pipeline",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = trainer.state.log_history[-2]['loss'] # último paso del entrenamiento
#perdida_final = resultado_entrenamiento.training_loss # promedio del entrenamiento
print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Se ejecuta el entranamiento del modelo  durante 30 epocas y se evalúa su rendimiento a través de la reducción de las función de perdida.

# Paso 3: Despliegue de Modelo Ajustado con LoRA

In [ ]:
modelo_lora.save_pretrained("/content/modelo_ajustado_lora")
print("Adaptador LoRA guardado en /content/modelo_ajustado_lora")
modelo_lora.push_to_hub("DanielIsildor/tinyllama-chatbot-IBEROPuebla-prospectos-lora")

Se guardan los resultados de entrenamiento y se publica el modelo entrenado en Huggin Face para futura reutilizacion.

# Pipeline completo RAG +  Fine Tuning

In [ ]:
!pip install groq -q

from groq import Groq
client = Groq(api_key=userdata.get('GROQ_API_KEY'))


def asistente(pregunta):
    fragmento = buscar_fragmento(pregunta)
    prompt = f"""Responde la pregunta del prospecto usando SOLO el siguiente lineamiento de becas. Si el lineamiento no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

for pregunta in [
    "¿Cuál es el apoyo máximo de beca?",
    "¿Soy egresado puedo acceder a una beca?",
    "¿Tengo un hermano estudiando, puedo acceder a una beca?",
]:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {asistente(pregunta)}\n")

Se compueba que las respuestas cumplen con los lineamientos de negocio.